## 8. CNN 网络的反向传播与参数更新

#### 1、这一节我们要解决什么问题 🎯

在前面的学习中，我们已经理解了：
- CNN 的基本结构（Conv 层、Pooling 层、全连接层）
- CNN 的前向传播过程
- 损失函数的计算方式（CrossEntropy、MSE 等）

但到这里，还缺少训练中最关键的一步：

**模型知道自己算错了之后，如何调整参数？**

这就需要引入：**反向传播（Backpropagation）与参数更新**

这一节我们要搞清楚：

1️⃣ **CNN 的反向传播和 MLP 有什么相同点？**  
2️⃣ **Conv 层在反向传播时，梯度是怎么计算的？**  
3️⃣ **Pooling 层没有参数，它的梯度怎么往前传？**  
4️⃣ **全连接层的梯度计算和 MLP 有什么关系？**  
5️⃣ **整个反向传播的流程是怎么串起来的？**  

**CNN 反向传播的核心仍然是链式法则；**  
**真正新的地方，在于 Conv 层和 Pooling 层这两种特殊结构，梯度的传递方式和 MLP 不一样。**

#### 2、先回顾：反向传播的核心思想 📦

在 MLP 中，我们已经学过反向传播的核心：**链式法则（Chain Rule）**

也就是说，对于一个复合函数：
```
L = f(g(x))
```
它对 x 的梯度是：
```
dL/dx = dL/df · df/dg · dg/dx
```

在神经网络中，这意味着：
- 从 Loss 出发
- 一层一层往前计算梯度
- 每一层都把"来自后面的梯度"乘以"本层的局部梯度"，再往前传

CNN 的反向传播本质上完全相同，只是：

**全连接层、Conv 层、Pooling 层的"局部梯度"计算方式各不相同。**

#### 3、CNN 的整体反向传播流程 🔄

##### 3.1 前向传播的路径（复习）
```
输入图像 → Conv层 → 激活函数 → Pooling层 → （重复）→ Flatten → 全连接层 → 输出 → Loss
```

##### 3.2 反向传播的路径（完全反过来）
```
Loss → 全连接层 → Flatten → Pooling层 → 激活函数 → Conv层 → （重复）→ 输入
```

也就是说：
- 梯度从 Loss 出发
- 先经过全连接层（和 MLP 完全一样）
- 经过 Flatten（只是恢复形状，梯度不变）
- 再经过 Pooling 层（梯度按规则分配）
- 再经过激活函数（逐元素计算局部梯度）
- 最后经过 Conv 层（计算对卷积核的梯度 + 继续往前传梯度）

每到一层，都需要做两件事：

**① 计算本层参数的梯度（用于更新参数）**  
**② 计算传给上一层的梯度（让前面的层继续更新）**

#### 4、全连接层的反向传播 🔧

##### 4.1 和 MLP 完全一样

全连接层的前向传播是：
```
z = x @ W.T + b
a = activation(z)
```

反向传播时，已知来自后面的梯度 δ（也就是 dL/da），需要计算：

- 对权重 W 的梯度：
```
dL/dW = δ.T @ x
```
- 对偏置 b 的梯度：
```
dL/db = δ（对 batch 求和）
```
- 传给前一层的梯度：
```
dL/dx = δ @ W
```

这些公式和 MLP 中完全相同，没有任何新内容。

##### 4.2 Flatten 层的梯度

Flatten 层只是把三维特征图 reshape 成一维向量，不做任何数值计算。

**梯度数值不变，只是把形状从一维还原回三维（恢复特征图的形状）。**

例如：
- Flatten 前特征图形状：`(B, C, H, W)`
- Flatten 后向量形状：`(B, C×H×W)`
- 反向传播时：梯度形状从 `(B, C×H×W)` 还原为 `(B, C, H, W)`

#### 5、Pooling 层的反向传播 🔹

Pooling 层没有任何可训练参数，所以反向传播时：

**不需要计算"对参数的梯度"，只需要把梯度往前传给 Conv 层。**

但具体怎么传，取决于 Pooling 的类型。

##### 5.1 Max Pooling 的反向传播

前向传播时，Max Pooling 在每个池化窗口内找到最大值，只保留它，其他值丢弃。

反向传播时：**梯度只传给前向传播中"赢得了最大值"的那个位置，其他位置梯度为 0。**

例如：
```
前向：窗口 [1, 3, 2, 4] → 最大值是 4，位于位置 3
反向：上游梯度 δ → 传回 [0, 0, 0, δ]
```

所以在前向传播时，Max Pooling 需要记录最大值的位置（称为"开关"或"mask"），反向传播时用它来分配梯度。

##### 5.2 Average Pooling 的反向传播

前向传播时对池化窗口内所有值求平均。

反向传播时：**上游梯度 δ 平均分配给窗口内的每一个位置。**

例如：
```
前向：窗口大小 2×2，求平均
反向：上游梯度 δ → 传回窗口内每个位置 δ/4
```

##### 5.3 核心理解

> "前向传播时，输出结果是怎么由输入得到的？那反向传播时，梯度就按同样的逻辑分配回去。"

- Max Pooling 只有一个元素"起作用"，所以梯度只给它
- Average Pooling 每个元素"均等起作用"，所以梯度均等分配


#### 6、激活函数的反向传播 ⚡

激活函数没有参数，它的反向传播是逐元素的局部梯度相乘。

##### 6.1 ReLU 的反向传播

前向传播：
```
a = ReLU(z) = max(0, z)
```

反向传播（局部梯度）：
```
da/dz = 1，如果 z > 0
da/dz = 0，如果 z ≤ 0
```

所以传给前面的梯度：
```
dL/dz = dL/da · da/dz
```
- 如果前向时 z > 0：梯度原样通过
- 如果前向时 z ≤ 0：梯度被截断为 0

##### 6.2 其他激活函数

sigmoid、tanh 等激活函数的反向传播方式完全相同，只是局部梯度公式不同：
- sigmoid：`da/dz = a(1-a)`
- tanh：`da/dz = 1 - a²`

CNN 中最常用的是 ReLU，所以重点掌握 ReLU 即可。

#### 7、Conv 层的反向传播（核心难点）🔸

Conv 层是 CNN 中真正独特的部分，它的反向传播和全连接层有本质区别。

##### 7.1 先回忆 Conv 层的前向传播

卷积层的前向传播，本质上是：

**卷积核（filter）在输入特征图上滑动，每次做一次局部区域的点积，得到一个输出值。**

用公式表示：
```
z[i,j] = Σ W[m,n] · x[i+m, j+n]（对所有卷积核位置求和）
```

这里：
- `x` 是输入特征图
- `W` 是卷积核（也就是这一层的参数）
- `z` 是输出特征图（激活前）

##### 7.2 反向传播需要求两个梯度

反向传播时，已知来自后面的梯度 δ（也就是 dL/dz），需要计算：

**（1）对卷积核 W 的梯度**（用于更新参数）
```
dL/dW[m,n] = Σ δ[i,j] · x[i+m, j+n]
```

也就是说：**对卷积核每个位置的梯度 = 上游梯度 δ 与输入 x 中对应局部区域做卷积。**

你可以这样理解：
- 前向传播时，卷积核在输入上滑动，做点积 → 得到输出
- 反向传播时，上游梯度在输入上滑动，做点积 → 得到卷积核的梯度

**（2）传给前一层的梯度**（让前面的层继续往前传）
```
dL/dx[i,j] = Σ δ[i-m, j-n] · W[m,n]
```

这个操作叫做：**转置卷积（或叫卷积核翻转后的卷积）**

形象地说：
- 前向传播时，卷积核正向滑动
- 反向传播时，卷积核翻转后滑动（旋转 180°），把上游梯度"散布"回输入的每个位置

##### 7.3 为什么 Conv 层的参数梯度"共享"了？

在卷积层中，同一个卷积核 W 会在输入特征图的多个位置上重复使用（滑动）。

这就意味着：**每个输出位置的梯度，都对同一个 W 的梯度有贡献。**

所以对 W 的最终梯度，需要把所有位置的贡献加起来：
```
dL/dW = Σ（所有滑动位置的梯度贡献）
```

这和前向传播时"参数共享"是对应的：
- 前向：同一个 W 在多个位置使用
- 反向：这多个位置对 W 的梯度都要累加

##### 7.4 核心理解

Conv 层反向传播的本质，仍然是链式法则，只是"局部梯度"的形式变成了卷积操作：
- 计算 `dL/dW`：用输入 x 和上游梯度做卷积 → 得到卷积核的梯度
- 计算 `dL/dx`：用翻转后的卷积核 W 和上游梯度做卷积 → 得到传给前一层的梯度


#### 8、参数更新：梯度下降 📉

##### 8.1 有哪些参数需要更新？

CNN 中需要更新的参数只有两类：
- Conv 层的卷积核权重 W 和偏置 b
- 全连接层的权重 W_fc 和偏置 b_fc

Pooling 层和激活函数层没有参数，不需要更新。

##### 8.2 参数更新的公式

和 MLP 完全一样，使用梯度下降：
```
W ← W - lr · dL/dW
b ← b - lr · dL/db
```

其中：
- `lr` 是学习率（learning rate）
- `dL/dW` 是通过反向传播计算得到的梯度

##### 8.3 在 PyTorch 中怎么实现？

好消息是：在 PyTorch 中，所有这些反向传播和参数更新的细节都被自动处理了。

我们只需要：
```python
loss.backward()   # 自动计算所有参数的梯度
optimizer.step()  # 按照梯度更新所有参数
```

PyTorch 会自动处理：
- Conv 层的卷积梯度计算
- Max Pooling 的 mask 记录与梯度分配
- 激活函数的局部梯度
- 全连接层的梯度

#### 9、本节总结 🧾
- CNN 反向传播的核心仍然是链式法则，方向是从 Loss 逐层往前传梯度
- 全连接层和 Flatten 层：和 MLP 完全相同
- Max Pooling：梯度只流向前向传播中最大值的那个位置
- Average Pooling：梯度均等分配给窗口内所有位置
- 激活函数（ReLU）：前向 z > 0 的位置梯度正常通过，z ≤ 0 的位置梯度为 0
- Conv 层：`dL/dW` 通过输入与上游梯度做卷积得到；`dL/dx` 通过翻转卷积核后与上游梯度做卷积得到
- 参数更新：只有 Conv 层和全连接层有参数，用 `W ← W - lr · dL/dW` 更新
- 在 PyTorch 中，`loss.backward()` + `optimizer.step()` 自动完成全部过程